General program

In [19]:
# Cell 1: Import dependencies and define Core Engine with Fault Analysis Tools
import numpy as np
import re
import os
import time
import json

class DiagnosticTensorSimulator:
    def __init__(self):
        self.inputs = []
        self.outputs = []
        self.gates = {}       
        self.topo_order = []  

    def load_bench(self, file_path: str):
        self.inputs = []
        self.outputs = []
        self.gates = {}
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"🚨 Path '{file_path}' does not exist.")
            
        with open(file_path, 'r') as f:
            lines = [line.strip() for line in f if line.strip() and not line.startswith('#')]
        
        for line in lines:
            if line.startswith("INPUT"):
                node = re.search(r'INPUT\s*\(\s*([\w\d_-]+)\s*\)', line).group(1)
                self.inputs.append(node)
            elif line.startswith("OUTPUT"):
                node = re.search(r'OUTPUT\s*\(\s*([\w\d_-]+)\s*\)', line).group(1)
                self.outputs.append(node)
            elif "=" in line:
                out_node, expr = line.split("=")
                out_node = out_node.strip()
                gate_type, ins_raw = expr.split("(")
                gate_type = gate_type.strip().upper()
                ins = [i.strip() for i in ins_raw.replace(")", "").split(",")]
                self.gates[out_node] = (gate_type, ins)

        # Topological Sort
        all_resolved = set(self.inputs)
        remaining_gates = dict(self.gates)
        self.topo_order = []
        
        while remaining_gates:
            ready_gates = [g for g, (gt, ins) in remaining_gates.items() if all(i in all_resolved for i in ins)]
            if not ready_gates:
                raise ValueError("🚨 Feedback loop or missing dependency discovered!")
            for g in ready_gates:
                self.topo_order.append(g)
                all_resolved.add(g)
                del remaining_gates[g]

    def get_collapsed_fault_list(self):
        """
        Generates a true local equivalence-collapsed stuck-at fault list.
        Applies standard gate collapsing rules (e.g., for NAND, input SA0s collapse to output SA1).
        """
        # Start by tracking all faults we potentially want to look at
        # We use a set to automatically handle shared nets/wires smoothly
        collapsed_faults = set()
        
        # 1. Add all faults for Primary Inputs as a baseline baseline
        for pi in self.inputs:
            collapsed_faults.add((pi, "SA0"))
            collapsed_faults.add((pi, "SA1"))
            
        # 2. Iterate through every gate and apply Boolean Equivalence rules
        for gate_out, (gate_type, gate_ins) in self.gates.items():
            # Always track both faults on the gate output to cover downstream paths
            collapsed_faults.add((gate_out, "SA0"))
            collapsed_faults.add((gate_out, "SA1"))
            
            if gate_type in ["NAND", "AND"]:
                # Rule: Input SA0 faults are equivalent to Output SA0 (for AND) or Output SA1 (for NAND)
                # Therefore, we can REMOVE/COLLAPSE all input SA0 faults from our tracking list!
                for inp in gate_ins:
                    # Keep input SA1 (it is distinct), but discard input SA0
                    collapsed_faults.add((inp, "SA1"))
                    if (inp, "SA0") in collapsed_faults and inp not in self.inputs:
                        collapsed_faults.remove((inp, "SA0"))
                        
            elif gate_type in ["NOR", "OR"]:
                # Rule: Input SA1 faults are equivalent to Output SA1 (for OR) or Output SA0 (for NOR)
                # Therefore, we can REMOVE/COLLAPSE all input SA1 faults!
                for inp in gate_ins:
                    # Keep input SA0, discard input SA1
                    collapsed_faults.add((inp, "SA0"))
                    if (inp, "SA1") in collapsed_faults and inp not in self.inputs:
                        collapsed_faults.remove((inp, "SA1"))
                        
            elif gate_type == "NOT":
                # Rule: Input SA0 == Output SA1, Input SA1 == Output SA0
                # We can completely drop internal NOT inputs if they aren't PIs
                for inp in gate_ins:
                    if inp not in self.inputs:
                        collapsed_faults.discard((inp, "SA0"))
                        collapsed_faults.discard((inp, "SA1"))
            else:
                # Fallback for complex gates (XOR/XNOR) where static local collapsing doesn't apply cleanly
                for inp in gate_ins:
                    collapsed_faults.add((inp, "SA0"))
                    collapsed_faults.add((inp, "SA1"))

        # Convert back to a clean, unique sorted list of tuples
        return sorted(list(collapsed_faults))

    def simulate(self, input_vectors: np.ndarray, fault_node=None, fault_type=None) -> dict:
        num_vectors = input_vectors.shape[1]
        node_values = {}
        
        for idx, inp in enumerate(self.inputs):
            node_values[inp] = input_vectors[idx].copy()
            if fault_node == inp:
                node_values[inp] = np.zeros(num_vectors, dtype=np.uint8) if fault_type == 'SA0' else np.ones(num_vectors, dtype=np.uint8)

        for gate in self.topo_order:
            gate_type, ins = self.gates[gate]
            in_vals = [node_values[i] for i in ins]
            
            if gate_type == "NAND":
                res = ~(np.bitwise_and.reduce(in_vals)) & 1
            elif gate_type == "AND":
                res = np.bitwise_and.reduce(in_vals)
            elif gate_type == "OR":
                res = np.bitwise_or.reduce(in_vals)
            elif gate_type == "NOR":
                res = ~(np.bitwise_or.reduce(in_vals)) & 1
            elif gate_type == "XOR":
                res = np.bitwise_xor.reduce(in_vals)
            elif gate_type == "XNOR":
                res = ~(np.bitwise_xor.reduce(in_vals)) & 1
            elif gate_type == "NOT":
                res = ~in_vals[0] & 1
            elif gate_type in ["BUFF", "BUF"]:
                res = in_vals[0].copy()
            else:
                raise NotImplementedError(f"Gate type '{gate_type}' is unhandled.")
                
            if fault_node == gate:
                res = np.zeros(num_vectors, dtype=np.uint8) if fault_type == 'SA0' else np.ones(num_vectors, dtype=np.uint8)
                    
            node_values[gate] = res
            
        return {out: node_values[out] for out in self.outputs}

In [20]:
# Cell 2: Test Vector Utility Loader
def load_test_vectors(file_path: str, num_inputs: int, fallback_count=100) -> np.ndarray:
    if file_path and os.path.exists(file_path):
        vectors = []
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                binary_match = re.match(r'^([01]+)', line)
                if binary_match:
                    pi_bits = binary_match.group(1)
                    if len(pi_bits) != num_inputs:
                        raise ValueError(f"🚨 Vector bit length mismatch! Circuit expects {num_inputs} inputs.")
                    vectors.append([int(b) for b in pi_bits])
        if vectors:
            print(f"📖 Successfully loaded {len(vectors)} test vectors from '{file_path}'.")
            return np.array(vectors, dtype=np.uint8).T
            
    print(f"⚠️ File '{file_path}' not found. Generating {fallback_count} random patterns dynamically...")
    np.random.seed(42)
    return np.random.randint(0, 2, size=(num_inputs, fallback_count), dtype=np.uint8)

In [ ]:
# Cell 3: Automated Diagnostic Batch Testing & Log Generator
def run_automated_fault_assessment(circuit, input_matrix, log_filename="fault_simulation_diagnostics.log"):
    """
    Executes all collapsed faults, benchmarks evaluation time, 
    gathers advanced diagnostic metrics, and streams results to a structured log file.
    """
    # 1. Acquire Golden Reference Output
    golden_outputs = circuit.simulate(input_matrix)
    total_vectors = input_matrix.shape[1]
    
    # 2. Get unique checkpoint faults
    fault_list = circuit.get_collapsed_fault_list()
    print(f"🔬 Total unique collapsed faults to analyze: {len(fault_list)}")
    
    log_entries = []
    detected_faults_count = 0
    
    print(f"✍️ Simulating and writing records directly to: '{log_filename}'...")
    bench_start=time.perf_counter()
    with open(log_filename, "w") as log_file:
        # Write header line
        log_file.write("# === VLSI TENSOR SIMULATION DIAGNOSTIC REPORT ===\n")
        log_file.write(f"# Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file.write(f"# Total Test Vectors Evaluated: {total_vectors}\n")
        log_file.write(f"# Total PIs count: {len(circuit.inputs)},list of PIs:{circuit.inputs}\n")
        log_file.write(f"# Total POs count: {len(circuit.outputs)},list of POS: {circuit.outputs}\n")
        log_file.write(f"# Total fault list count: {len(fault_list)},list of fault list: {fault_list}\n\n")
        
        for f_node, f_type in fault_list:
            # Benchmark precisely using high-resolution performance timers
            start_time = time.perf_counter()
            faulty_outputs = circuit.simulate(input_matrix, fault_node=f_node, fault_type=f_type)
            end_time = time.perf_counter()
            
            duration_microseconds = (end_time - start_time) * 1_000_000
            
            # Metric Discovery: Compute detection vectors
            detection_mask = np.zeros(total_vectors, dtype=bool)
            corrupted_pos = []
            hamming_distance_per_po = {}
            
            for out in circuit.outputs:
                mismatch_mask = (golden_outputs[out] != faulty_outputs[out])
                detection_mask |= mismatch_mask
                
                mismatch_count = int(np.sum(mismatch_mask))
                if mismatch_count > 0:
                    corrupted_pos.append(out)
                    hamming_distance_per_po[out] = mismatch_count
            
            is_detected = bool(np.any(detection_mask))
            detect_indices = np.where(detection_mask)[0].tolist()
            
            if is_detected:
                detected_faults_count += 1
                
            # Build an enriched diagnostic payload record
            diagnostic_record = {
                "fault_node": f_node,
                "fault_type": f_type,
                "status": "DETECTED" if is_detected else "UNDETECTED",
                "duration_us": round(duration_microseconds, 8),
                "total_detecting_vectors": len(detect_indices),
                "detecting_vector_indices": detect_indices,
                "corrupted_primary_outputs": corrupted_pos,
                "output_mismatch_counts": hamming_distance_per_po, #sum of all results where golder po <> fault po
                "raw_golden_po": {out: golden_outputs[out].tolist() for out in circuit.outputs},
                "raw_faulty_po": {out: faulty_outputs[out].tolist() for out in circuit.outputs}
            }
            
            # Write row directly to disk as JSON string lines for easy streaming parsing later
            log_file.write(json.dumps(diagnostic_record) + "\n")
            log_entries.append(diagnostic_record)
        bench_end=time.perf_counter()
        duration_bench_us = (bench_end - bench_start) * 1_000_000
        log_file.write(f"{'*'*100}\nCoverage: {(detected_faults_count / len(fault_list)) * 100}%\n")
        log_file.write(f"{'*'*100}\nTotal time: {duration_bench_us}")
    # Calculate Summary Stats
    coverage = (detected_faults_count / len(fault_list)) * 100
    print(f"🎉 Analysis Complete! Overall Fault Coverage: {coverage:.2f}%")
    return log_entries

In [22]:
# Cell 4: Production Run
# BENCH_PATH = "./data.nogit/c17.bench"
# VECTOR_PATH = "./data.nogit/c17.tests"
# LOG_PATH = "./logs/c17_diagnostics_report.log"
def benchmark(bench,test,log):
    # Initialize and run
    circuit = DiagnosticTensorSimulator()
    circuit.load_bench(bench)

    input_matrix = load_test_vectors(test, num_inputs=len(circuit.inputs), fallback_count=100)

    # Run full evaluation
    records = run_automated_fault_assessment(circuit, input_matrix, log_filename=log)

    # # Preview the first log record inside your notebook display
    # print("\n👀 Sample JSON Log Entry Preview (First Fault Tested):")
    # print(json.dumps(records[0], indent=4))

In [23]:
import os
bench_file=[]
test_file=[]
log_file=[]
for a in os.listdir('./data.nogit/'):
    if ".bench" in a:
        bench_file.append('./data.nogit/'+a)
        log_file.append('./logs/'+a.split('.')[0]+'_report.log')
    else:
        test_file.append('./data.nogit/'+a)
bench_file.sort()
test_file.sort()
log_file.sort()

for a,b,c in zip(bench_file,test_file,log_file):
    print(a,'\t',b,'\t',c)
    try:
        benchmark(a,b,c)
    except:
        print(f"{a} failed, pending to run benchmark again")

./data.nogit/c1355.bench 	 ./data.nogit/c1355.tests 	 ./logs/c1355_report.log
📖 Successfully loaded 100 test vectors from './data.nogit/c1355.tests'.
🔬 Total unique collapsed faults to analyze: 692
✍️ Simulating and writing records directly to: './logs/c1355_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 91.04%
./data.nogit/c17.bench 	 ./data.nogit/c17.tests 	 ./logs/c17_report.log
📖 Successfully loaded 100 test vectors from './data.nogit/c17.tests'.
🔬 Total unique collapsed faults to analyze: 18
✍️ Simulating and writing records directly to: './logs/c17_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 100.00%
./data.nogit/c1908.bench 	 ./data.nogit/c1908.tests 	 ./logs/c1908_report.log
📖 Successfully loaded 100 test vectors from './data.nogit/c1908.tests'.
🔬 Total unique collapsed faults to analyze: 948
✍️ Simulating and writing records directly to: './logs/c1908_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 82.49%
./data.nogit/c2670.bench 	 ./dat

In [24]:
BENCH_PATH = "./data.nogit/c17.bench"
VECTOR_PATH = "./data.nogit/c17.tests"
LOG_PATH = "./logs/c17_diagnostics_report.log"
benchmark(BENCH_PATH,VECTOR_PATH,LOG_PATH)

📖 Successfully loaded 100 test vectors from './data.nogit/c17.tests'.
🔬 Total unique collapsed faults to analyze: 18
✍️ Simulating and writing records directly to: './logs/c17_diagnostics_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 100.00%


In [ ]:
import json
import os
import re

def generate_summary_from_header(log_filepath, bench_name="Circuit"):
    """
    Parses metadata directly from the structured log file header,
    then aggregates runtime and fault coverage data from the log body.
    """
    if not os.path.exists(log_filepath):
        print(f"🚨 Log file '{log_filepath}' not found.")
        return None

    # Header tracking elements
    pi_count = 0
    po_count = 0
    fault_list_count = 0
    
    # Body calculations
    detected_faults = 0
    cumulative_time_us = 0.0

    with open(log_filepath, "r") as f:
        for line in f:
            line = line.strip()
            
            # 1. Parse Metadata Metrics out of the Header Comments
            if line.startswith("#"):
                if "Total PIs count:" in line:
                    match = re.search(r'Total PIs count:\s*(\d+)', line)
                    if match: pi_count = int(match.group(1))
                elif "Total POs count:" in line:
                    match = re.search(r'Total POs count:\s*(\d+)', line)
                    if match: po_count = int(match.group(1))
                elif "Total fault list count:" in line:
                    match = re.search(r'Total fault list count:\s*(\d+)', line)
                    if match: fault_list_count = int(match.group(1))
                continue
                
            # 2. Parse Execution Records from the Data Rows
            if not line:
                continue
                
            try:
                record = json.loads(line)
                cumulative_time_us += record.get("duration_us", 0.0)
                if record.get("status") == "DETECTED":
                    detected_faults += 1
            except json.JSONDecodeError:
                continue

    # Final Metric Balancing Calculations
    fault_coverage = (detected_faults / fault_list_count * 100) if fault_list_count > 0 else 0.0
    total_time_ms = cumulative_time_us / 1000.0
    properties_str = f"{pi_count} PIs / {po_count} POs"
    
    # Build cleanly formatted ASCII Output Block
    header_str = f"| {'Circuit':<12} | {'Properties':<16} | {'Fault List Count':<18} | {'Total Time':<14} | {'Fault Coverage':<14} |"
    divider = "+" + "-"*14 + "+" + "-"*18 + "+" + "-"*20 + "+" + "-"*16 + "+" + "-"*16 + "+"
    #row_str = f"| {bench_name:<12} | {properties_str:<16} | {fault_list_count:<18} | {total_time_ms:.2f} ms:<14}| {fault_coverage:.2f}%:<14} |"
    
    # Correct string slicing for uniform row cells
    row_str = f"| {bench_name:<12} | {properties_str:<16} | {fault_list_count:<18} | {f'{total_time_ms:.2f} ms':<14} | {f'{fault_coverage:.2f}%':<14} |"

    print("\n📊 === VLSI FAULT SIMULATION BASELINE SUMMARY ===")
    print(divider)
    print(header_str)
    print(divider)
    print(row_str)
    print(divider)

🚨 Log file 'c17_diagnostics_report.log' not found.


In [37]:
# --- EXECUTION ---
# Path to your generated log file from Cell 4
def summary(LOG_PATH):
    # LOG_PATH = "./logs/c499_report.log"
    name=LOG_PATH.split('/')[-1].split('_')[0]
    # Parse the logs and render the finalized table 
    # (Pass the expected PI count manually to complete the properties metric)
    stats = generate_summary_from_header(LOG_PATH, bench_name=name)
    # print_summary_table(stats, pi_count_override=5)

In [38]:
for c in log_file:
    print(c)
    summary(c)

./logs/c1355_report.log

📊 === VLSI FAULT SIMULATION BASELINE SUMMARY ===
+--------------+------------------+--------------------+----------------+----------------+
| Circuit      | Properties       | Fault List Count   | Total Time     | Fault Coverage |
+--------------+------------------+--------------------+----------------+----------------+
| c1355        | 41 PIs / 32 POs  | 692                | 1998.00 ms     | 91.04%         |
+--------------+------------------+--------------------+----------------+----------------+
./logs/c17_report.log

📊 === VLSI FAULT SIMULATION BASELINE SUMMARY ===
+--------------+------------------+--------------------+----------------+----------------+
| Circuit      | Properties       | Fault List Count   | Total Time     | Fault Coverage |
+--------------+------------------+--------------------+----------------+----------------+
| c17          | 5 PIs / 2 POs    | 18                 | 0.85 ms        | 100.00%        |
+--------------+------------------+